# Logistics Regression Implementation

Logistics regression classification:
    Uses Big Five traits to predict Low/Medium/High CWB and OCB.

Algorithm idea:
    Logistic Regression is a linear classification model that estimates the probability of class membership based on a weighted combination of predictors. For multiclass outcomes, it separates categories by learning decision boundaries that best distinguish between classes.

Evaluation metrics:
    - Higher accuracy
    - Higher precision
    - Higher recall
    - Higher macro F1

## Import dataset and data preprocessing

Preprocess data to transform big five and organizational behavioral measures.

In [2]:
# import predetermined functions from src/
import sys
import os

sys.path.append(os.path.abspath("../../src"))

from ml_models.logistic_regression import run_logistic_regression_classification

In [4]:
# import and preprocess data
import json
from pathlib import Path
import pandas as pd

# Find repo root automatically
repo_root = Path.cwd().resolve()
while not (repo_root / "BFI_2_life_narative_metadata.json").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find BFI_2_life_narative_metadata.json")
    repo_root = repo_root.parent

# Load data
with open(repo_root / "BFI_2_life_narative_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(repo_root / "BFI_2_life_narrative.json", "r", encoding="utf-8") as f:
    responses = json.load(f)

df = pd.DataFrame(responses)

# Reverse-code BFI items
reverse_items = metadata["reverse_code_items"]

for item in reverse_items:
    if item in df.columns:
        df[item] = 6 - df[item]

# Big Five trait means
traits = {
    key: value
    for key, value in metadata.items()
    if (
        isinstance(value, list)
        and value
        and isinstance(value[0], str)
        and value[0].startswith("Item")
        and not key.startswith(("Item", "Q", "CWB", "OCB"))
        and key != "reverse_code_items"
    )
}

for trait, items in traits.items():
    available_items = [item for item in items if item in df.columns]
    df[trait] = df[available_items].mean(axis=1)

# CWB and OCB means
cwb_cols = [f"CWB{i}" for i in range(1, 11) if f"CWB{i}" in df.columns]
ocb_cols = [f"OCB{i}" for i in range(1, 11) if f"OCB{i}" in df.columns]

df["CWB"] = df[cwb_cols].mean(axis=1)
df["OCB"] = df[ocb_cols].mean(axis=1)

# Targets
big_five = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]

print("✓ Data preprocessing complete")
print("Repo root:", repo_root)
print("df shape:", df.shape)
df.head()

✓ Data preprocessing complete
Repo root: /Users/cindy/cmor438_Spring2026/cmor438_Spring2026
df shape: (500, 138)


,ID,Gender,Race,Age,Q1,Q2,Q3,Q4,Q5,Q6,...,Intellectual_Curiosity,Aesthetic_Sensitivity,Creative_Imagination,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness,CWB,OCB
0,1,Woman,White,22,"chicago, illinois; in a suburb near chicago. i...",i always worked very hard and did my best. i h...,i had multiple teachers that were influential ...,art or reading/writing. i am a really creative...,"math, because it doesn't always come as easily...",one of my heroes has always been my mom. she a...,...,4.50,4.75,5.00,2.416667,4.250000,3.083333,4.500000,4.750000,1.7,3.2
1,2,Woman,White,38,I am from VA and you? I grew up in TX and it w...,A very hardworking and a brilliant student in ...,I always liked my Mathematics teacher very muc...,I loved mathematics and biology as I loved to ...,Probably geography was my least favorite for l...,For me it was always Einstein as he was an awe...,...,3.50,3.75,4.25,4.583333,4.250000,4.250000,1.583333,3.833333,1.1,2.7
2,3,Woman,White,19,"I'm from Wichita,Kansas My life was great grow...",I was horrible in school. I was diagnosed with...,Oh yeah! I had a great teacher senior year of ...,My favorite subject in school was probably His...,I hated English. I was never good at it.,my heros were probably my parents. They were a...,...,3.50,4.25,4.50,3.916667,3.916667,2.500000,3.500000,4.083333,2.1,3.6
3,4,Man,White,21,"northbrook, IL; I grew up in the northern subu...",I was a great student. I was in Honors and AP ...,I definitely had influential teachers. My AP p...,Math because there was a definitive answer and...,History because it was so boring to me alwya,My mom for sure and brothers,...,3.50,3.00,3.25,4.666667,4.500000,3.083333,2.583333,3.250000,1.5,2.8
4,5,Woman,White,26,"I am from Denver Colorado, I grew up in Cherry...",I was a very bright student in school and ever...,My mathematics teacher was influtial in my pro...,Mathematics was my favorite subject because i ...,History was my worst subject because the class...,My parents were my heroes because they gave me...,...,3.75,3.25,3.25,2.916667,3.250000,3.583333,2.916667,3.416667,1.5,2.8


## Run logistics regression

In [5]:
# define predictors and outcomes
predictors = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]

outcomes = ["CWB", "OCB"]

In [6]:
# run logistic regression
logistic_results = run_logistic_regression_classification(
    df=df,
    predictors=predictors,
    outcomes=outcomes,
    test_size=0.2,
    random_state=42
)

In [7]:
# view metrics
for outcome, result in logistic_results.items():
    print(f"\n{outcome}")
    print("Accuracy:", round(result["accuracy"], 4))
    print("Precision:", round(result["precision_macro"], 4))
    print("Recall:", round(result["recall_macro"], 4))
    print("Macro F1:", round(result["f1_macro"], 4))
    print(result["classification_report"])


CWB
Accuracy: 0.5
Precision: 0.5
Recall: 0.4959
Macro F1: 0.4825
              precision    recall  f1-score   support

        High       0.50      0.55      0.52        31
         Low       0.50      0.67      0.57        36
      Medium       0.50      0.27      0.35        33

    accuracy                           0.50       100
   macro avg       0.50      0.50      0.48       100
weighted avg       0.50      0.50      0.48       100


OCB
Accuracy: 0.4
Precision: 0.3525
Recall: 0.3776
Macro F1: 0.3507
              precision    recall  f1-score   support

        High       0.36      0.38      0.37        32
         Low       0.46      0.66      0.54        38
      Medium       0.23      0.10      0.14        30

    accuracy                           0.40       100
   macro avg       0.35      0.38      0.35       100
weighted avg       0.36      0.40      0.37       100



## Results analysis

Logistic regression was also used to classify participants into **Low, Medium, or High** levels of CWB and OCB using the Big Five personality traits.

### CWB Classification

| Metric | Value |
|---|---:|
| Accuracy | 0.50 |
| Precision (Macro) | 0.500 |
| Recall (Macro) | 0.496 |
| F1 (Macro) | 0.482 |

For CWB, logistic regression achieved **50% accuracy**, substantially higher than the perceptron and clearly above the 33% chance baseline.

Class-level results showed the strongest performance for the **Low CWB** group (F1 = 0.57), followed by **High CWB** (F1 = 0.52). Performance was weaker for the **Medium** group (F1 = 0.35).

The model appears better at distinguishing individuals at the lower and higher ends of counterproductive behavior than those in the middle range.

### OCB Classification

| Metric | Value |
|---|---:|
| Accuracy | 0.40 |
| Precision (Macro) | 0.352 |
| Recall (Macro) | 0.378 |
| F1 (Macro) | 0.351 |

For OCB, logistic regression achieved **40% accuracy**, only slightly better than the perceptron.

The strongest performance was for the **Low OCB** group (F1 = 0.54), while the model struggled considerably with the **Medium OCB** group (F1 = 0.14).

This suggests that personality traits were more useful for identifying lower levels of citizenship behavior than moderate levels.

### Interpretation

Logistic regression demonstrated better overall performance than the perceptron, particularly for CWB classification. Its probability-based framework likely provided more stable and effective decision boundaries than the perceptron’s simpler update-based rule. However, the findings suggest that Big Five personality traits contain useful predictive information, but classification performance remains modest. Additional predictors beyond personality may be needed to substantially improve workplace behavior classification.